# 🧠 ResumeIQ AI — Module 2: Hiring Recommendation Model

**Problem:** Given a scored resume, predict the hiring outcome:
> `Reject` → `Maybe` → `Interview` → `Strong Hire`

**Dataset:** Synthetically generated training data derived from the ResumeIQ AI scoring pipeline.
Each row is a feature vector representing one analyzed resume.

**Model:** XGBoost Multiclass Classifier with SHAP Explainability

**Metrics:** Accuracy · Weighted F1 · ROC-AUC (OvR) · Confusion Matrix · SHAP Feature Importance

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, roc_auc_score
)
import xgboost as xgb

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('SHAP not installed. Run: pip install shap')

import os
os.makedirs('plots', exist_ok=True)
os.makedirs('../data/models', exist_ok=True)

print('Libraries loaded ✓')
print(f'XGBoost version: {xgb.__version__}')

## Step 1 — Generate Training Dataset

We generate 10,000 synthetic samples using realistic distributions for each feature.
Labels are assigned via a deterministic scoring rule that mirrors how real recruiters evaluate candidates,
then injected with 15% label noise to prevent overfitting.

In [ ]:
np.random.seed(42)
N = 10_000

FEATURE_NAMES = [
    'semantic_match',        # 0: resume vs JD cosine similarity (0–1)
    'evidence_score',        # 1: cross-section evidence for skills (0–1)
    'experience_quality',    # 2: impact/scale/metrics in bullets (0–1)
    'project_complexity',    # 3: avg project complexity score (0–1)
    'skill_confidence',      # 4: confidence in claimed skills (0–1)
    'hallucination_rate',    # 5: fraction of unsupported skill claims (0–1)
    'keyword_stuffing_risk', # 6: probability of keyword stuffing (0–1)
    'skill_breadth',         # 7: normalised skill count (0–1)
    'experience_count',      # 8: normalised number of roles (0–1)
    'project_count',         # 9: normalised number of projects (0–1)
    'missing_required',      # 10: normalised missing required skills (0–1)
    'total_tenure',          # 11: normalised total months of experience (0–1)
]

# Generate correlated features using a realistic covariance structure
def generate_candidate_profile(n):
    # Base quality level drives most features
    quality = np.random.beta(2, 2, n)  # 0–1, peak at 0.5
    noise = lambda scale: np.random.normal(0, scale, n)
    clip = lambda x: np.clip(x, 0.0, 1.0)
    
    semantic_match       = clip(quality * 0.8 + noise(0.15))
    evidence_score       = clip(quality * 0.75 + noise(0.15))
    experience_quality   = clip(quality * 0.7  + noise(0.18))
    project_complexity   = clip(quality * 0.65 + noise(0.20))
    skill_confidence     = clip(quality * 0.72 + noise(0.15))
    hallucination_rate   = clip((1 - quality) * 0.4 + noise(0.10))  # inversely correlated
    keyword_stuffing     = clip((1 - quality) * 0.3 + noise(0.10))
    skill_breadth        = clip(quality * 0.6  + noise(0.20))
    experience_count     = clip(quality * 0.5  + noise(0.25))
    project_count        = clip(quality * 0.55 + noise(0.20))
    missing_required     = clip((1 - quality) * 0.5 + noise(0.15))
    total_tenure         = clip(quality * 0.6  + noise(0.20))
    
    return np.column_stack([
        semantic_match, evidence_score, experience_quality, project_complexity,
        skill_confidence, hallucination_rate, keyword_stuffing, skill_breadth,
        experience_count, project_count, missing_required, total_tenure
    ])

X = generate_candidate_profile(N)
df = pd.DataFrame(X, columns=FEATURE_NAMES)

print(f'Generated dataset: {df.shape}')
print(df.describe().round(3))

In [ ]:
# Label generation: deterministic rule-based scoring
def assign_label(row):
    score = (
        row['semantic_match']      * 0.30
        + row['evidence_score']    * 0.20
        + row['experience_quality']* 0.15
        + row['project_complexity']* 0.10
        + row['skill_confidence']  * 0.10
        - row['hallucination_rate']* 0.08
        - row['keyword_stuffing_risk'] * 0.05
        - row['missing_required']  * 0.07
        + row['total_tenure']      * 0.05
    )
    if score >= 0.72:   return 3  # Strong Hire
    elif score >= 0.52: return 2  # Interview
    elif score >= 0.35: return 1  # Maybe
    else:               return 0  # Reject

y_clean = df.apply(assign_label, axis=1).values

# Add 12% label noise (realistic uncertainty in hiring decisions)
noise_mask = np.random.random(N) < 0.12
y_noisy = y_clean.copy()
y_noisy[noise_mask] = np.random.randint(0, 4, noise_mask.sum())

y = y_noisy
LABEL_NAMES = ['Reject', 'Maybe', 'Interview', 'Strong Hire']

print('Label distribution:')
for i, name in enumerate(LABEL_NAMES):
    count = (y == i).sum()
    print(f'  {name:12s}: {count:5d} ({count/N*100:.1f}%)')

## Step 2 — EDA

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()
colors = ['#ef4444', '#f59e0b', '#3b82f6', '#22c55e']

for i, feat in enumerate(FEATURE_NAMES):
    for label_idx, label_name in enumerate(LABEL_NAMES):
        mask = y == label_idx
        axes[i].hist(X[mask, i], bins=25, alpha=0.5, color=colors[label_idx],
                     label=label_name, density=True)
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=9)
    axes[i].set_xlabel('Value')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', ncol=2, fontsize=9)
fig.suptitle('Feature Distributions by Hiring Outcome', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
corr = pd.DataFrame(X, columns=FEATURE_NAMES).corr()
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/02_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Train XGBoost Classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)
print('\nModel trained ✓')

## Step 4 — Evaluation

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc    = accuracy_score(y_test, y_pred)
f1_w   = f1_score(y_test, y_pred, average='weighted')
f1_m   = f1_score(y_test, y_pred, average='macro')
roc    = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

print('=== XGBoost Hiring Recommender — Evaluation ===')
print(f'Test Accuracy     : {acc:.4f}  ({acc*100:.2f}%)')
print(f'Weighted F1       : {f1_w:.4f}')
print(f'Macro F1          : {f1_m:.4f}')
print(f'ROC-AUC (OvR)     : {roc:.4f}')
print(f'CV Accuracy (5-fold): {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print()
print(classification_report(y_test, y_pred, target_names=LABEL_NAMES, digits=4))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
ax.set_title('Confusion Matrix — XGBoost Hiring Recommender', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('plots/02_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — Feature Importance + SHAP

In [ ]:
# XGBoost native feature importance
importance = model.feature_importances_
sorted_idx = np.argsort(importance)

fig, ax = plt.subplots(figsize=(9, 6))
colors_bar = ['#22c55e' if FEATURE_NAMES[i] not in ['hallucination_rate', 'keyword_stuffing_risk', 'missing_required']
              else '#ef4444' for i in sorted_idx]
ax.barh([FEATURE_NAMES[i].replace('_', ' ').title() for i in sorted_idx],
        importance[sorted_idx], color=colors_bar)
ax.set_xlabel('Importance Score')
ax.set_title('XGBoost Feature Importance — Hiring Recommender', fontweight='bold')
plt.tight_layout()
plt.savefig('plots/02_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Feature importance ranking:')
for i in sorted_idx[::-1]:
    print(f'  {FEATURE_NAMES[i]:28s}: {importance[i]:.4f}')

In [ ]:
if SHAP_AVAILABLE:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test[:200])
    
    print('SHAP Summary Plot (Strong Hire class):')
    shap.summary_plot(
        shap_values[3], X_test[:200],
        feature_names=[f.replace('_', ' ').title() for f in FEATURE_NAMES],
        show=False
    )
    plt.title('SHAP Values — Strong Hire Class', fontweight='bold')
    plt.tight_layout()
    plt.savefig('plots/02_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Install shap for SHAP analysis: pip install shap')

## Step 6 — Save Model

In [ ]:
bundle = {
    'model': model,
    'feature_names': FEATURE_NAMES,
    'label_names': LABEL_NAMES,
    'accuracy': float(acc),
    'f1_weighted': float(f1_w),
    'f1_macro': float(f1_m),
    'roc_auc': float(roc),
}

with open('../data/models/hiring_recommender.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print('Model saved → data/models/hiring_recommender.pkl')
print(f'Accuracy: {acc*100:.2f}% | ROC-AUC: {roc:.4f} | F1 Weighted: {f1_w:.4f}')

## Summary

| Metric | Value |
|---|---|
| Model | XGBoost Multiclass (Softprob) |
| Dataset | 10,000 synthetic samples with realistic feature distributions |
| Classes | Reject / Maybe / Interview / Strong Hire |
| Features | 12 (semantic match, evidence, quality, complexity, confidence, …) |
| Train/Test | 80% / 20% stratified |
| Label Noise | 12% realistic uncertainty |

**This model powers Engine 5 (Recruiter Decision Engine) in ResumeIQ AI.**

**Why XGBoost?**
- Handles mixed feature types natively
- Natively supports SHAP explainability (TreeExplainer)
- Robust to noisy labels via gradient boosting
- Outperforms deep models on tabular data at this scale